
# Gaussian Inference Basics

In the Representation chapter, we built a reusable `Gaussian` class for

$$
\mathbf{X}\sim\mathcal{N}(\boldsymbol{\mu},\Sigma).
$$

We now use that representation to study the fundamental operations required for inference in Gaussian models.

The main questions are the same as in discrete inference:

1. **Marginalization**  
   What is the distribution of a subset of variables?

2. **Conditioning**  
   What should we believe about some variables after observing others?

3. **Product of Gaussians**  
   How can two Gaussian sources of information about the same variable be combined?

A special property of Gaussian models makes these operations particularly useful:

> Marginalization, conditioning, and multiplication preserve Gaussian structure.

Therefore, many Gaussian inference problems can be solved analytically by manipulating mean vectors and covariance matrices rather than enumerating states or drawing samples.

## Reuse the Gaussian Class

In [31]:

%run ../../../01_Representation/01_Probability_Distribution/03_Multivariate_Probability_Distribution/Gaussian_Class.ipynb

# 1. From Discrete Inference to Gaussian Inference

Before studying Gaussian inference, let us recall what inference meant in discrete probabilistic models.

Suppose we have a joint distribution

$$
P(X,Y).
$$

There are two fundamental inference questions:

1. **Marginalization**

   What is the probability of $X$ regardless of the value of $Y$?

   $$
   P(X)=
   \sum_Y P(X,Y).
   $$

2. **Conditioning**

   What is the probability of $X$ after observing $Y=y$?

   $$
   P(X\mid Y=y).
   $$

These two operations formed the basis of all exact inference algorithms studied previously.

Now consider a continuous random vector

$$
\mathbf{x}=
\begin{bmatrix}
\mathbf{x}_1\\
\mathbf{x}_2
\end{bmatrix},
$$

whose joint distribution is Gaussian,

$$
\mathbf{x}
\sim
\mathcal N(\boldsymbol{\mu},\Sigma).
$$

The inference questions remain exactly the same:

- What is the distribution of $\mathbf{x}_1$?
- What is the distribution of $\mathbf{x}_1$ after observing $\mathbf{x}_2$?

The difference lies only in how the uncertainty is represented.

Instead of manipulating probability tables, we manipulate the mean vector and covariance matrix of a Gaussian distribution.

One of the remarkable properties of Gaussian distributions is that both marginalization and conditioning produce another Gaussian distribution. This allows many inference problems to be solved analytically without approximation.

# 2. Joint Gaussian Distribution

Consider a Gaussian random vector

$$\mathbf{x}=
\begin{bmatrix}
\mathbf{x}_1\\
\mathbf{x}_2
\end{bmatrix}
\sim
\mathcal N
\left(
\begin{bmatrix}
\boldsymbol{\mu}_1\\
\boldsymbol{\mu}_2
\end{bmatrix},
\begin{bmatrix}
\Sigma_{11} & \Sigma_{12}\\
\Sigma_{21} & \Sigma_{22}
\end{bmatrix}
\right).
$$

The mean vector is partitioned into two groups corresponding to the variables $\mathbf{x}_1$ and $\mathbf{x}_2$.

Similarly, the covariance matrix is partitioned into four blocks:

- $\Sigma_{11}$: covariance within $\mathbf{x}_1$,
- $\Sigma_{22}$: covariance within $\mathbf{x}_2$,
- $\Sigma_{12}$: covariance between $\mathbf{x}_1$ and $\mathbf{x}_2$,
- $\Sigma_{21}$: transpose of $\Sigma_{12}$.

All Gaussian inference operations will be derived from this partitioned representation.

# 3. Gaussian Marginalization

In the previous chapter, marginalization was introduced as the process of eliminating variables that are not relevant to the current query.

Suppose we have a joint probability distribution

$$
P(X,Y).
$$

If our goal is to determine the probability distribution of $X$ alone, then the variable $Y$ becomes a hidden variable. To remove its influence, we sum over all possible values of $Y$:

$$
P(X)=\sum_Y P(X,Y).
$$

For continuous random variables, the idea remains exactly the same. The only difference is that the hidden variable can take infinitely many values. Therefore, summation is replaced by integration:

$$
p(x)=\int p(x,y)\,dy.
$$

Although the mathematical operation changes from summation to integration, the underlying objective is unchanged:

> Remove the variables that are not part of the query while preserving the uncertainty of the variables that remain.

Now consider a jointly Gaussian random vector

$$\mathbf{x}=
\begin{bmatrix}
\mathbf{x}_1\\
\mathbf{x}_2
\end{bmatrix}
\sim
\mathcal N
\left(
\begin{bmatrix}
\boldsymbol{\mu}_1\\
\boldsymbol{\mu}_2
\end{bmatrix},
\begin{bmatrix}
\Sigma_{11} & \Sigma_{12}\\
\Sigma_{21} & \Sigma_{22}
\end{bmatrix}
\right).
$$

Suppose we are interested only in the variables contained in $\mathbf{x}_1$. Formally, this means computing

$$p(\mathbf{x}_1)=
\int
p(\mathbf{x}_1,\mathbf{x}_2)
\,d\mathbf{x}_2.
$$

At first glance, this appears to require evaluating a multidimensional integral. Fortunately, Gaussian distributions possess a remarkable property.

The marginal distribution of a multivariate Gaussian is itself Gaussian.

Even more surprisingly, the marginal distribution can be obtained without explicitly evaluating the integral.

The result is simply

$$
\boxed{
\mathbf{x}_1
\sim
\mathcal N
(
\boldsymbol{\mu}_1,
\Sigma_{11}
)
}
$$

In other words, marginalization consists of selecting the components of the mean vector and covariance matrix that correspond to the variables of interest.

No numerical integration is required.

## 3.1 Why Does This Work?

Recall that the mean vector stores the expected value of every variable,

$$\boldsymbol{\mu}=
\begin{bmatrix}
\boldsymbol{\mu}_1\\
\boldsymbol{\mu}_2
\end{bmatrix}.
$$

Similarly, the covariance matrix stores how every pair of variables varies together.

When we marginalize out $\mathbf{x}_2$, we are no longer interested in describing those variables.

Therefore:

- the mean values associated with $\mathbf{x}_2$ are discarded,
- the covariances involving $\mathbf{x}_2$ are also discarded.

The remaining variables already contain all the information required to describe the marginal distribution.

Consequently, the marginal Gaussian is completely specified by

- the corresponding entries of the mean vector, and
- the corresponding block of the covariance matrix.


## 3.2 Numerical Example

Consider

$$
\mathbf{x}=
\begin{bmatrix}
X\\
Y\\
Z
\end{bmatrix}
$$

with

$$
\boldsymbol{\mu}=
\begin{bmatrix}
1\\
3\\
5
\end{bmatrix}
$$

and

$$
\Sigma=
\begin{bmatrix}
2.0 & 0.8 & 0.3\\
0.8 & 1.5 & 0.6\\
0.3 & 0.6 & 1.2
\end{bmatrix}.
$$

Suppose we only care about $X$ and $Z$.

We keep indices `[0, 2]`.


In [32]:

joint = Gaussian(
    mean=np.array([1.0, 3.0, 5.0]),
    covariance=np.array([
        [2.0, 0.8, 0.3],
        [0.8, 1.5, 0.6],
        [0.3, 0.6, 1.2],
    ]),
)

indices = np.array([0, 2])

marginal_mean = joint.mean[indices]

marginal_covariance = joint.covariance[
    np.ix_(indices, indices)
]

print("Marginal mean:")
print(marginal_mean)

print("\nMarginal covariance:")
print(marginal_covariance)

Marginal mean:
[1. 5.]

Marginal covariance:
[[2.  0.3]
 [0.3 1.2]]



## 3.3 Method Implementation

Now that the mathematics is clear, write the operation as a standalone method.

It will later become part of `GaussianInference`.

In [33]:

def marginal(
    self,
    indices: list[int] | np.ndarray,
) -> "GaussianInference":
    indices = np.asarray(
        indices,
        dtype=int,
    )

    if indices.ndim != 1:
        raise ValueError(
            "indices must be one-dimensional."
        )

    if len(indices) == 0:
        raise ValueError(
            "At least one variable must be retained."
        )

    if len(np.unique(indices)) != len(indices):
        raise ValueError(
            "indices must not contain duplicates."
        )

    if np.any(indices < 0) or np.any(
        indices >= self.dimension
    ):
        raise ValueError(
            "indices contain an invalid variable index."
        )

    marginal_mean = self.mean[indices]

    marginal_covariance = self.covariance[
        np.ix_(indices, indices)
    ]

    return GaussianInference(
        mean=marginal_mean,
        covariance=marginal_covariance,
    )



## 3.4 Verification with Samples

If the analytical marginal is correct, samples from the joint Gaussian should have the same mean and covariance after the eliminated dimensions are discarded.

In [43]:
rng = np.random.default_rng(42)

samples = joint.sample(
    n_samples=100_000,
    rng=rng,
)

xz_samples = samples[:, [0, 2]]

print("Empirical mean:")
print(xz_samples.mean(axis=0))

print("\nEmpirical covariance:")
print(
    np.cov(
        xz_samples,
        rowvar=False,
    )
)


Empirical mean:
[1.00220571 5.00443421]

Empirical covariance:
[[2.00771408 0.30184931]
 [0.30184931 1.19948757]]



### Marginalization Summary

| Model | Operation |
|---|---|
| Discrete | Sum over eliminated variables |
| General continuous | Integrate over eliminated variables |
| Gaussian | Select the relevant mean and covariance blocks |



# 4. Gaussian Conditioning

## 4.1 Motivation

Marginalization answers:

> What do I believe about a subset of variables if I ignore the others?

Conditioning asks:

> What do I believe about some variables after observing other variables?

Suppose

$$
\begin{bmatrix}
\mathbf{x}_1\\
\mathbf{x}_2
\end{bmatrix}
\sim
\mathcal{N}
\left(
\begin{bmatrix}
\boldsymbol{\mu}_1\\
\boldsymbol{\mu}_2
\end{bmatrix},
\begin{bmatrix}
\Sigma_{11} & \Sigma_{12}\\
\Sigma_{21} & \Sigma_{22}
\end{bmatrix}
\right).
$$

After observing

$$
\mathbf{x}_2=\mathbf{a},
$$

the conditional distribution remains Gaussian:

$$
\mathbf{x}_1
\mid
\mathbf{x}_2=\mathbf{a}
\sim
\mathcal{N}
(
\boldsymbol{\mu}_{1|2},
\Sigma_{1|2}
).
$$

The conditional mean is

$$
\boxed{\boldsymbol{\mu}_{1|2}=
\boldsymbol{\mu}_1+
\Sigma_{12}
\Sigma_{22}^{-1}
(\mathbf{a}-
\boldsymbol{\mu}_2
)
}
$$

and the conditional covariance is

$$
\boxed{
\Sigma_{1|2}
=\Sigma_{11}-
\Sigma_{12}
\Sigma_{22}^{-1}
\Sigma_{21}
}
$$



## 4.2 Understanding the Mean Update

The term

$$
\mathbf{a}-\boldsymbol{\mu}_2
$$

is the difference between the observation and what the model expected.

The term

$$
\Sigma_{12}\Sigma_{22}^{-1}
$$

determines how strongly that difference should influence the variables of interest.

So conditioning has the intuitive form

$$\boxed{
\text{new mean}
=\text{old mean}
+
\text{correction}
}
$$

This structure will appear again later in the Kalman filter.



## 4.3 Understanding the Covariance Update

The conditional covariance is

$$
\Sigma_{1|2}=
\Sigma_{11}-
\Sigma_{12}
\Sigma_{22}^{-1}
\Sigma_{21}.
$$

A positive-semidefinite quantity is removed from the original covariance.

Conceptually:

> An informative observation reduces uncertainty.

If the observed and queried variables are weakly related, the reduction is small.

If they are strongly related, observing one provides substantial information about the other.



## 4.4 Worked Example

Consider

$$
\begin{bmatrix}
X\\
Y
\end{bmatrix}
\sim
\mathcal{N}
\left(
\begin{bmatrix}
0\\
0
\end{bmatrix},
\begin{bmatrix}
4 & 3\\
3 & 4
\end{bmatrix}
\right).
$$

Suppose

$$
Y=2.
$$

Because $X$ and $Y$ have positive covariance, observing a positive $Y$ should move our belief about $X$ toward positive values.


In [35]:

joint_xy = Gaussian(
    mean=np.array([0.0, 0.0]),
    covariance=np.array([
        [4.0, 3.0],
        [3.0, 4.0],
    ]),
)

query_indices = np.array([0])
evidence_indices = np.array([1])
evidence_values = np.array([2.0])

mu_q = joint_xy.mean[query_indices]
mu_e = joint_xy.mean[evidence_indices]

Sigma_qq = joint_xy.covariance[
    np.ix_(query_indices, query_indices)
]

Sigma_qe = joint_xy.covariance[
    np.ix_(query_indices, evidence_indices)
]

Sigma_eq = joint_xy.covariance[
    np.ix_(evidence_indices, query_indices)
]

Sigma_ee = joint_xy.covariance[
    np.ix_(evidence_indices, evidence_indices)
]

innovation = evidence_values - mu_e

conditional_mean = (
    mu_q
    + Sigma_qe
    @ np.linalg.solve(
        Sigma_ee,
        innovation,
    )
)

conditional_covariance = (
    Sigma_qq
    - Sigma_qe
    @ np.linalg.solve(
        Sigma_ee,
        Sigma_eq,
    )
)


In [36]:
print("Conditional mean:")
print(conditional_mean)

print("\nConditional covariance:")
print(conditional_covariance)


Conditional mean:
[1.5]

Conditional covariance:
[[1.75]]



## 4.5 Method Implementation


In [37]:

def condition(
    self,
    query_indices: list[int] | np.ndarray,
    evidence_indices: list[int] | np.ndarray,
    evidence_values: np.ndarray,
) -> "GaussianInference":
    query_indices = np.asarray(
        query_indices,
        dtype=int,
    )

    evidence_indices = np.asarray(
        evidence_indices,
        dtype=int,
    )

    evidence_values = np.asarray(
        evidence_values,
        dtype=float,
    )

    if query_indices.ndim != 1:
        raise ValueError(
            "query_indices must be one-dimensional."
        )

    if evidence_indices.ndim != 1:
        raise ValueError(
            "evidence_indices must be one-dimensional."
        )

    if evidence_values.ndim != 1:
        raise ValueError(
            "evidence_values must be one-dimensional."
        )

    if len(query_indices) == 0:
        raise ValueError(
            "At least one query variable is required."
        )

    if len(evidence_indices) == 0:
        raise ValueError(
            "At least one evidence variable is required."
        )

    if len(evidence_indices) != len(evidence_values):
        raise ValueError(
            "Each evidence variable must have one value."
        )

    all_indices = np.concatenate([
        query_indices,
        evidence_indices,
    ])

    if len(np.unique(all_indices)) != len(all_indices):
        raise ValueError(
            "Query and evidence indices must be unique "
            "and must not overlap."
        )

    if np.any(all_indices < 0) or np.any(
        all_indices >= self.dimension
    ):
        raise ValueError(
            "An invalid variable index was provided."
        )

    mu_q = self.mean[query_indices]
    mu_e = self.mean[evidence_indices]

    Sigma_qq = self.covariance[
        np.ix_(query_indices, query_indices)
    ]

    Sigma_qe = self.covariance[
        np.ix_(query_indices, evidence_indices)
    ]

    Sigma_eq = self.covariance[
        np.ix_(evidence_indices, query_indices)
    ]

    Sigma_ee = self.covariance[
        np.ix_(evidence_indices, evidence_indices)
    ]

    innovation = evidence_values - mu_e

    conditional_mean = (
        mu_q
        + Sigma_qe
        @ np.linalg.solve(
            Sigma_ee,
            innovation,
        )
    )

    conditional_covariance = (
        Sigma_qq
        - Sigma_qe
        @ np.linalg.solve(
            Sigma_ee,
            Sigma_eq,
        )
    )

    return GaussianInference(
        mean=conditional_mean,
        covariance=conditional_covariance,
    )



## 4.6 Geometric Interpretation

For a two-dimensional Gaussian:

- **marginalization** is like projecting the probability cloud onto one axis,
- **conditioning** is like slicing the probability cloud at the observed value.

This distinction is fundamental.

Marginalization removes variables.

Conditioning incorporates information about variables.



# 5. Product of Gaussian Distributions

## 5.1 Motivation

Suppose two independent sources provide information about the same unknown variable.

For example:

- one localization system estimates position,
- another sensor provides a second estimate.

Let

$$
p_1(\mathbf{x})=
\mathcal{N}
(
\boldsymbol{\mu}_1,
\Sigma_1
)
$$

and

$$
p_2(\mathbf{x})=
\mathcal{N}
(
\boldsymbol{\mu}_2,
\Sigma_2
).
$$

Their product is proportional to another Gaussian:

$$
p_1(\mathbf{x})
p_2(\mathbf{x})
\propto
\mathcal{N}
(
\boldsymbol{\mu},
\Sigma
).
$$

The proportionality sign appears because the raw product of two probability densities is not automatically normalized.



## 5.2 Precision Form

Define the precision matrix

$$
\Lambda=
\Sigma^{-1}.
$$

For two Gaussian beliefs,

$$
\Lambda=
\Lambda_1
+
\Lambda_2.
$$

Therefore,

$$
\boxed{
\Sigma=
(
\Sigma_1^{-1}
+
\Sigma_2^{-1}
)^{-1}
}
$$

and

$$
\boxed{
\boldsymbol{\mu}=
\Sigma
\left(
\Sigma_1^{-1}\boldsymbol{\mu}_1
+
\Sigma_2^{-1}\boldsymbol{\mu}_2
\right)
}
$$

This means that more precise information contributes more strongly to the combined estimate.



## 5.3 Worked Example

Suppose

$$
p_1(x)=
\mathcal{N}(0,4)
$$

and

$$
p_2(x)=
\mathcal{N}(3,1).
$$

The second belief is more precise because its variance is smaller.

Therefore, the combined mean should lie closer to \(3\) than to \(0\).


In [38]:

belief_1 = Gaussian(
    mean=np.array([0.0]),
    covariance=np.array([[4.0]]),
)

belief_2 = Gaussian(
    mean=np.array([3.0]),
    covariance=np.array([[1.0]]),
)

precision_1 = np.linalg.inv(
    belief_1.covariance
)

precision_2 = np.linalg.inv(
    belief_2.covariance
)

combined_covariance = np.linalg.inv(
    precision_1 + precision_2
)

combined_mean = combined_covariance @ (
    precision_1 @ belief_1.mean
    + precision_2 @ belief_2.mean
)




In [39]:
print("Combined mean:")
print(combined_mean)

print("\nCombined covariance:")
print(combined_covariance)

Combined mean:
[2.4]

Combined covariance:
[[0.8]]



## 5.4 Method Implementation


In [40]:

def multiply(
    self,
    other: "GaussianInference",
) -> "GaussianInference":
    if not isinstance(other, Gaussian):
        raise TypeError(
            "other must be a Gaussian."
        )

    if self.dimension != other.dimension:
        raise ValueError(
            "Both Gaussians must have the same dimension."
        )

    precision_self = np.linalg.inv(
        self.covariance
    )

    precision_other = np.linalg.inv(
        other.covariance
    )

    combined_precision = (
        precision_self
        + precision_other
    )

    combined_covariance = np.linalg.inv(
        combined_precision
    )

    combined_mean = combined_covariance @ (
        precision_self @ self.mean
        + precision_other @ other.mean
    )

    return GaussianInference(
        mean=combined_mean,
        covariance=combined_covariance,
    )



# 6. Final Implementation

We have now understood and implemented each operation independently.

We can combine them into a single class that extends the representation-level `Gaussian` class.

This gives a clean separation:

```text
Gaussian
    │
    │ representation
    ▼
GaussianInference
    │
    │ inference operations
    ├── marginal()
    ├── condition()
    └── multiply()
```


In [41]:

class GaussianInference(Gaussian):
    """
    Gaussian distribution with basic inference operations.

    Extends the representation-level Gaussian class with:

    - marginalization,
    - conditioning,
    - multiplication of compatible Gaussian beliefs.
    """

    def marginal(
        self,
        indices: list[int] | np.ndarray,
    ) -> "GaussianInference":
        indices = np.asarray(
            indices,
            dtype=int,
        )

        if indices.ndim != 1:
            raise ValueError(
                "indices must be one-dimensional."
            )

        if len(indices) == 0:
            raise ValueError(
                "At least one variable must be retained."
            )

        if len(np.unique(indices)) != len(indices):
            raise ValueError(
                "indices must not contain duplicates."
            )

        if np.any(indices < 0) or np.any(
            indices >= self.dimension
        ):
            raise ValueError(
                "indices contain an invalid variable index."
            )

        marginal_mean = self.mean[indices]

        marginal_covariance = self.covariance[
            np.ix_(indices, indices)
        ]

        return GaussianInference(
            mean=marginal_mean,
            covariance=marginal_covariance,
        )

    def condition(
        self,
        query_indices: list[int] | np.ndarray,
        evidence_indices: list[int] | np.ndarray,
        evidence_values: np.ndarray,
    ) -> "GaussianInference":
        query_indices = np.asarray(
            query_indices,
            dtype=int,
        )

        evidence_indices = np.asarray(
            evidence_indices,
            dtype=int,
        )

        evidence_values = np.asarray(
            evidence_values,
            dtype=float,
        )

        if query_indices.ndim != 1:
            raise ValueError(
                "query_indices must be one-dimensional."
            )

        if evidence_indices.ndim != 1:
            raise ValueError(
                "evidence_indices must be one-dimensional."
            )

        if evidence_values.ndim != 1:
            raise ValueError(
                "evidence_values must be one-dimensional."
            )

        if len(query_indices) == 0:
            raise ValueError(
                "At least one query variable is required."
            )

        if len(evidence_indices) == 0:
            raise ValueError(
                "At least one evidence variable is required."
            )

        if len(evidence_indices) != len(evidence_values):
            raise ValueError(
                "Each evidence variable must have one value."
            )

        all_indices = np.concatenate([
            query_indices,
            evidence_indices,
        ])

        if len(np.unique(all_indices)) != len(all_indices):
            raise ValueError(
                "Query and evidence indices must be unique "
                "and must not overlap."
            )

        if np.any(all_indices < 0) or np.any(
            all_indices >= self.dimension
        ):
            raise ValueError(
                "An invalid variable index was provided."
            )

        mu_q = self.mean[query_indices]
        mu_e = self.mean[evidence_indices]

        Sigma_qq = self.covariance[
            np.ix_(query_indices, query_indices)
        ]

        Sigma_qe = self.covariance[
            np.ix_(query_indices, evidence_indices)
        ]

        Sigma_eq = self.covariance[
            np.ix_(evidence_indices, query_indices)
        ]

        Sigma_ee = self.covariance[
            np.ix_(evidence_indices, evidence_indices)
        ]

        innovation = evidence_values - mu_e

        conditional_mean = (
            mu_q
            + Sigma_qe
            @ np.linalg.solve(
                Sigma_ee,
                innovation,
            )
        )

        conditional_covariance = (
            Sigma_qq
            - Sigma_qe
            @ np.linalg.solve(
                Sigma_ee,
                Sigma_eq,
            )
        )

        return GaussianInference(
            mean=conditional_mean,
            covariance=conditional_covariance,
        )

    def multiply(
        self,
        other: Gaussian,
    ) -> "GaussianInference":
        if not isinstance(other, Gaussian):
            raise TypeError(
                "other must be a Gaussian."
            )

        if self.dimension != other.dimension:
            raise ValueError(
                "Both Gaussians must have the same dimension."
            )

        precision_self = np.linalg.inv(
            self.covariance
        )

        precision_other = np.linalg.inv(
            other.covariance
        )

        combined_precision = (
            precision_self
            + precision_other
        )

        combined_covariance = np.linalg.inv(
            combined_precision
        )

        combined_mean = combined_covariance @ (
            precision_self @ self.mean
            + precision_other @ other.mean
        )

        return GaussianInference(
            mean=combined_mean,
            covariance=combined_covariance,
        )
